# EDA of Raw Real Estate Data

In this notebook, we will analyze the raw real estate dataset collected by the Airflow DAG and stored in the PostgreSQL (table `mle1_real_estate_dataset`).

The goal of this analysis is to identify data quality issues and prepare the cleaning logic that will be used to create the cleaned dataset for further model training.

## 1. Data Loading and Overview

First, let's import the necessary libraries, load environment variables, and prepare the tools for connecting to the PostgreSQL database.

In [1]:
# disable warnings to keep the output clean
import warnings
warnings.filterwarnings("ignore")

# import libraries for working with environment variables and file paths
import os
from pathlib import Path

# import libraries for data manipulation and numerical computations
import pandas as pd
import numpy as np

# import libraries for loading configuration and creating a PostgreSQL connection
from dotenv import load_dotenv
from sqlalchemy import create_engine

# import PostgreSQL driver for direct database connection
import psycopg2

In [2]:
# configure table display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

Now we will load the environment variables from the local .env file and create a PostgreSQL connection. Sensitive values such as passwords are not displayed in the notebook.

In [3]:
# define the project directory and path to the local .env file
project_dir = Path.cwd().parents[1]
env_path = project_dir / ".env"

In [4]:
# load environment variables from the .env file
load_dotenv(env_path)

True

In [5]:
# read PostgreSQL connection parameters from environment variables
db_user = os.getenv("DB_DESTINATION_USER")
db_password = os.getenv("DB_DESTINATION_PASSWORD")
db_host = os.getenv("DB_DESTINATION_HOST")
db_port = os.getenv("DB_DESTINATION_PORT")
db_name = os.getenv("DB_DESTINATION_NAME")

In [6]:
# create a SQL-Alchemy engine for connecting to PostgreSQL
connection_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_url)

In [7]:
# check that the .env file was found and the engine was created
print("Environment file found:", env_path.exists())

Environment file found: True


The PostgreSQL connection was created successfully.

Next we will load the raw real estate dataset from the PostgreSQL table `mle1_real_estate_dataset` and check its basic structure.

In [8]:
# define the source table name
raw_table_name = "mle1_real_estate_dataset"

In [9]:
# create a direct PostgreSQL connection using environment variables
connection = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password
)

In [10]:
# load the dataset from PostgreSQL into a pandas dataframe
df_raw = pd.read_sql(f"SELECT * FROM {raw_table_name};", connection)

In [11]:
# close the database connection after loading the data
connection.close()

Now let's look at the general structure of the dataset: the number of rows and columns, the first records, and the basic information about data types and missing values.

In [12]:
# display the basic dataset shape
print("Dataset shape:", df_raw.shape)

Dataset shape: (141362, 18)


In [13]:
# display the first five rows of the dataset
display(df_raw.head())

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.90,1,False,False,35.10,9500000,1965,6,55.72,37.78,2.64,84,12,True
1,1,18012,7,0.00,16.60,1,False,False,43.00,13500000,2001,2,55.79,37.61,3.00,97,10,True
2,2,17821,9,9.00,32.00,2,False,False,56.00,13500000,2000,4,55.74,37.76,2.70,80,10,True
3,3,18579,1,10.10,43.10,3,False,False,76.00,20000000,2002,4,55.67,37.57,2.64,771,17,True
4,4,9293,3,3.00,14.00,1,False,False,24.00,5200000,1971,1,55.81,37.71,2.60,208,9,True


In [14]:
# display general information about columns, data types, and non-null values
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141362 entries, 0 to 141361
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flat_id            141362 non-null  int64  
 1   building_id        141362 non-null  int64  
 2   floor              141362 non-null  int64  
 3   kitchen_area       141362 non-null  float64
 4   living_area        141362 non-null  float64
 5   rooms              141362 non-null  int64  
 6   is_apartment       141362 non-null  bool   
 7   studio             141362 non-null  bool   
 8   total_area         141362 non-null  float64
 9   price              141362 non-null  int64  
 10  build_year         141362 non-null  int64  
 11  building_type_int  141362 non-null  int64  
 12  latitude           141362 non-null  float64
 13  longitude          141362 non-null  float64
 14  ceiling_height     141362 non-null  float64
 15  flats_count        141362 non-null  int64  
 16  fl

The dataset contains 141362 rows and 18 columns. There are no missing values, but we will perform an additional check to confirm this.

## 2. Missing Values

In [15]:
# calculate the number of missing values in each column
missing_values = df_raw.isna().sum().sort_values(ascending=False)
missing_values

flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64

Indeed, the dataset does not contain missing values: the number of `NaN` values is zero for all features. However, the absence of `NaN` values does not necessarily mean that the data is fully clean. 

Next, we need to check for duplicates, as well as possible implicit errors and outliers in the numerical features.

## 3. Duplicates

In [16]:
# check the number of fully duplicated rows
full_duplicates_count = df_raw.duplicated().sum()

In [17]:
# check the number of duplicates by apartment identifier 'flat_id'
flat_id_duplicates_count = df_raw.duplicated(subset=["flat_id"]).sum()

In [18]:
# check the results
print(f"Number of fully duplicated rows: {full_duplicates_count}")
print(f"Number of duplicates by flat_id: {flat_id_duplicates_count}")

Number of fully duplicated rows: 0
Number of duplicates by flat_id: 0


No fully duplicated rows were found in the dataset. There are also no duplicates by `flat_id`, which means that each apartment is represented only once in the data. However, we will still implement duplicate removal as a separate cleaning function to make the Airflow DAG more robust and reusable.

## 4. EDA of Numerical Features

Now let's look at descriptive statistics for the numerical features. This will help us identify implicit data errors and potential outliers.

In [19]:
# select numerical columns
numeric_columns = df_raw.select_dtypes(include=["int64", "float64"]).columns

In [20]:
# display descriptive statistics for numerical features
df_raw[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
flat_id,141362.00,70680.50,40807.84,0.00,35340.25,70680.50,106020.75,141361.00
building_id,141362.00,14053.67,6988.83,1.00,8535.25,14332.00,20475.00,24620.00
floor,141362.00,7.47,5.72,1.00,3.00,6.00,10.00,56.00
kitchen_area,141362.00,9.00,5.26,0.00,6.10,8.80,10.20,203.00
living_area,141362.00,31.06,23.97,0.00,19.00,29.40,41.40,700.00
rooms,141362.00,2.13,0.99,1.00,1.00,2.00,3.00,20.00
total_area,141362.00,62.37,40.30,11.00,39.30,53.00,72.00,960.30
price,141362.00,19441619.95,66269544.30,11.00,8900000.00,11850000.00,16950000.00,9873737728.00
build_year,141362.00,1986.60,22.14,1901.00,1969.00,1985.00,2007.00,2023.00
building_type_int,141362.00,3.23,1.46,0.00,2.00,4.00,4.00,6.00


The descriptive statistics show several potential anomalies and outliers. Special attention should be paid to features such as `price`, `kitchen_area`, `living_area`, `total_area`, and `ceiling_height`. Some values look unlikely at first glance, for example a price of `11` or a ceiling height of `27`. However, not all extreme values should be automatically treated as errors. Some of them may represent rare but valid properties, such as apartments with a very large area, a large kitchen, or a very high price.

More reliable indicators of data errors are logical contradictions: very low, zero, or negative prices, zero or negative area values, a floor number higher than the total number of floors, and unrealistic ceiling heights.

For ceiling_height = `27`, we can assume that this may be a scale error where `27` actually means `2.7`. Therefore, instead of deleting such rows, we can correct ceiling height values in the range from `20 to 60` by dividing them by `10`.

In [21]:
# check the number of clearly incorrect or suspicious values
suspicious_values = {
"price <= 0": (df_raw["price"] <= 0).sum(),
"price < 100000": (df_raw["price"] < 100000).sum(),
"total_area <= 0": (df_raw["total_area"] <= 0).sum(),
"kitchen_area < 0": (df_raw["kitchen_area"] < 0).sum(),
"living_area < 0": (df_raw["living_area"] < 0).sum(),
"rooms <= 0": (df_raw["rooms"] <= 0).sum(),
"floor <= 0": (df_raw["floor"] <= 0).sum(),
"floors_total <= 0": (df_raw["floors_total"] <= 0).sum(),
"floor > floors_total": (df_raw["floor"] > df_raw["floors_total"]).sum(),
"ceiling_height < 2": (df_raw["ceiling_height"] < 2).sum(),
"ceiling_height > 6": (df_raw["ceiling_height"] > 6).sum(),
}
pd.Series(suspicious_values).sort_values(ascending=False)

price < 100000          145
ceiling_height > 6       11
price <= 0                0
total_area <= 0           0
kitchen_area < 0          0
rooms <= 0                0
living_area < 0           0
floor <= 0                0
floors_total <= 0         0
floor > floors_total      0
ceiling_height < 2        0
dtype: int64

The check identified two main groups of suspicious values: apartments with `price < 100000` and apartments with `ceiling_height > 6`. Let's inspect these rows separately.

In [22]:
# inspect apartments with suspiciously low prices
low_price_rows = df_raw[df_raw["price"] < 100000].sort_values("price")
print(f"Number of rows with price < 100000: {low_price_rows.shape[0]}")
display(low_price_rows.head(10))

Number of rows with price < 100000: 145


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
69633,69633,20672,9,6.00,33.00,2,False,False,42.00,11,2007,4,55.72,37.91,2.78,368,24,True
107198,107198,9797,2,0.00,21.00,2,False,False,70.00,19,1972,1,55.80,37.73,2.64,288,9,True
104573,104573,5381,9,8.00,46.00,1,False,False,59.00,85,1964,1,55.81,37.71,2.64,108,9,True
60417,60417,22183,3,0.00,0.00,1,False,False,36.00,1500,2012,4,55.68,37.67,2.80,180,10,True
61931,61931,3393,3,6.00,18.00,1,False,False,30.00,2000,1960,1,55.85,37.45,2.48,79,5,False
76800,76800,6439,4,9.00,20.00,1,False,False,38.00,2000,1966,1,55.84,37.49,2.48,198,5,False
128961,128961,20428,6,8.00,26.00,1,False,False,40.00,2000,2007,2,55.67,37.76,2.90,169,9,True
126489,126489,12495,5,8.00,19.60,1,False,False,38.20,2300,1978,4,55.90,37.57,2.64,127,16,True
137162,137162,6439,4,9.00,20.00,1,False,False,40.00,2400,1966,1,55.84,37.49,2.48,198,5,False
59840,59840,14009,12,10.00,26.00,1,True,False,36.00,2500,1983,4,55.90,37.59,2.64,512,24,True


These rows clearly look like data errors. Prices such as `11, 19, 85, 1500, and 2000` do not look realistic for apartments with normal area and property characteristics. Since price is the target variable, we cannot reliably restore the correct value. Therefore, during data cleaning, we will remove rows where `price < 100000`.

Now let's check the second group of suspicious values: unusually high ceiling heights.

In [23]:
# inspect apartments with suspiciously high ceiling heights
high_ceiling_rows = df_raw[df_raw["ceiling_height"] > 6].sort_values("ceiling_height")
print(f"Number of rows with ceiling_height > 6: {high_ceiling_rows.shape[0]}")
display(high_ceiling_rows)

Number of rows with ceiling_height > 6: 11


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
15762,15762,631,5,18.30,84.00,3,False,False,132.20,34500000,1931,1,55.78,37.71,7.00,25,5,True
51006,51006,632,5,14.20,67.80,4,False,False,108.20,35000000,1931,1,55.78,37.71,7.00,25,6,True
50624,50624,632,5,14.20,67.80,4,False,False,108.20,32950000,1931,1,55.78,37.71,7.00,25,6,True
121050,121050,631,1,8.00,35.00,2,False,False,54.00,12300000,1931,1,55.78,37.71,7.00,25,5,True
21022,21022,12052,5,5.80,0.00,2,False,False,44.70,11000000,1977,4,55.66,37.71,8.00,80,5,True
9572,9572,12052,3,5.50,46.00,3,False,False,55.20,7200000,1977,4,55.66,37.71,8.00,80,5,True
44301,44301,12052,4,6.00,37.40,3,False,False,55.50,7000000,1977,4,55.66,37.71,8.00,80,5,True
7486,7486,12052,4,6.00,37.40,3,False,False,55.50,7050000,1977,4,55.66,37.71,8.00,80,5,True
41975,41975,12052,5,5.80,0.00,2,False,False,44.70,11100000,1977,4,55.66,37.71,8.00,80,5,True
121759,121759,12052,5,5.80,29.00,2,False,False,44.70,10700000,1977,4,55.66,37.71,8.00,80,5,True


The dataset contains `145` rows with suspiciously low prices where `price < 100000`. These values look like data errors because the apartments have normal areas and characteristics, but the prices are unrealistically low. Since price is the target variable, we will remove these rows during cleaning.

The dataset also contains `11` rows with `ceiling_height > 6`. Values such as `7 or 8` may be rare but still potentially possible for some real estate objects, especially in older buildings. Therefore, we will not remove them automatically.

However, the value `27` looks like a scale error and most likely means `2.7`. During cleaning, we will correct `ceiling_height` values in the range from `20 to 60` by dividing them by `10`.

## 5. Data Cleaning Functions

And now we will create cleaning functions based on the issues identified during EDA. These functions will remove possible duplicates, filter out rows with incorrect target values, and correct ceiling height scale errors.

In [24]:
# remove fully duplicated rows and duplicates by apartment identifier 'flat_id'
def remove_duplicates(data):
    data = data.copy()
    data = data.drop_duplicates()
    data = data.drop_duplicates(subset=["flat_id"])
    return data

In [25]:
# remove rows with invalid prices
def remove_invalid_price(data):
    data = data.copy()
    data = data[data["price"] >= 100000]
    return data

In [26]:
# fix ceiling height values that look like scale errors
def fix_ceiling_height(data):
    data = data.copy()
    # values between 20 and 60 most likely represent meters multiplied by 10
    mask = data["ceiling_height"].between(20, 60)
    data.loc[mask, "ceiling_height"] = data.loc[mask, "ceiling_height"] / 10  
    return data

In [27]:
# apply all cleaning functions step by step
def clean_data(data):
    data = data.copy()
    data = remove_duplicates(data)
    data = remove_invalid_price(data)
    data = fix_ceiling_height(data)
    return data

In [28]:
# apply the cleaning functions to the raw dataset
df_clean = clean_data(df_raw)

In [29]:
# compare the dataset size before and after cleaning
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Number of removed rows: {df_raw.shape[0] - df_clean.shape[0]}")

Raw dataset shape: (141362, 18)
Cleaned dataset shape: (141217, 18)
Number of removed rows: 145


In [30]:
# check how the row with the original ceiling height of 27 changed after cleaning
df_clean[df_clean["flat_id"] == 83673][["flat_id", "ceiling_height"]]

,flat_id,ceiling_height
83673,83673,2.70


After applying the cleaning functions, the dataset size changed from `141362 to 141217` rows. In total, `145` rows with incorrect price values below `100000` were removed.

The scale issue in the `ceiling_height` feature was also corrected: the value `27` was converted to `2.7`. The values `7 and 8` were kept unchanged, as they may refer to rare but potentially valid real estate objects.

And now let's perform the final check of the cleaned dataset.

In [31]:
# check the cleaned dataset after applying the cleaning functions
final_checks = {
    "Fully duplicated rows": df_clean.duplicated().sum(),
    "Duplicates by flat_id": df_clean.duplicated(subset=["flat_id"]).sum(),
    "Missing values": df_clean.isna().sum().sum(),
    "Rows with price < 100000": (df_clean["price"] < 100000).sum(),
    "Rows with ceiling_height >= 20": (df_clean["ceiling_height"] >= 20).sum(),
}
pd.Series(final_checks)

Fully duplicated rows             0
Duplicates by flat_id             0
Missing values                    0
Rows with price < 100000          0
Rows with ceiling_height >= 20    0
dtype: int64

After applying the cleaning functions, there are no rows left with an incorrect price below `100000` and no `ceiling_height` values greater than or equal to `20`. We also performed additional checks for missing values and fully duplicated rows, and none were found. 

As a result, the cleaned dataset contains `141217` rows and `18` columns. These functions can be reused in the data cleaning DAG at the next step.

In [32]:
# display descriptive statistics for numerical features after cleaning
df_clean[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
flat_id,141217.00,70666.16,40810.08,0.00,35320.00,70663.00,106011.00,141361.00
building_id,141217.00,14056.02,6989.03,1.00,8541.00,14335.00,20477.00,24620.00
floor,141217.00,7.47,5.72,1.00,3.00,6.00,10.00,56.00
kitchen_area,141217.00,9.00,5.26,0.00,6.10,8.80,10.20,203.00
living_area,141217.00,31.06,23.98,0.00,19.00,29.40,41.40,700.00
rooms,141217.00,2.13,0.99,1.00,1.00,2.00,3.00,20.00
total_area,141217.00,62.39,40.31,11.00,39.30,53.00,72.00,960.30
price,141217.00,19461556.32,66300636.06,100000.00,8950000.00,11890000.00,16950000.00,9873737728.00
build_year,141217.00,1986.61,22.14,1901.00,1969.00,1985.00,2007.00,2023.00
building_type_int,141217.00,3.23,1.46,0.00,2.00,4.00,4.00,6.00


After cleaning, the minimum price is `100000`, and the maximum ceiling height is now `8`. Other high values, for example in area, number of rooms, or number of floors, were not removed because they may represent rare real estate objects. At this stage, we treat them as outliers that the model should take into account rather than as errors that must be removed.

## 6. Conclusion

At this stage, we analyzed the raw dataset `mle1_real_estate_dataset` created during the first part of the project and prepared the cleaning logic.

First, we imported the required libraries and configured the connection to the PostgreSQL database. Then, we loaded the `mle1_real_estate_dataset` table from the database. The dataset contains `141362` rows and `18` columns.

During the initial analysis, we found that the dataset does not contain explicit missing values. We also checked for duplicates and found no fully duplicated rows and no duplicates by the apartment identifier `flat_id`.

Next, we analyzed the numerical features. The descriptive statistics revealed several suspicious values: `145` rows with an unusually low price (`price < 100000`) and one value of `ceiling_height = 27`, which appears to be a scale-related error.

To clean the dataset, we created the following functions:
- `remove_duplicates()` removes fully duplicated rows and duplicates by `flat_id`.
- `remove_invalid_price()` removes rows with invalid prices where `price < 100000`.
- `fix_ceiling_height()` corrects ceiling height values between `20 and 60` by dividing them by `10`.
- `clean_data()` applies all cleaning steps sequentially.

After applying the cleaning functions, the dataset size changed from `141362 to 141217` rows. In total, `145` rows with invalid prices were removed, and the ceiling height value of `27` was corrected to `2.7`. The final validation confirmed that the cleaned dataset contains no fully duplicated rows, no duplicates by `flat_id`, no explicit missing values, no rows with `price < 100000`, and no ceiling height values greater than or equal to `20`.

The next step is to transfer this cleaning logic to an Airflow DAG that will create a cleaned table in the PostgreSQL database.